# Laboratorio 01: Búsqueda de Grover (2 qubits)

El algoritmo de Grover localiza un elemento marcado en una base no ordenada de N=4 elementos en **O(√N)** pasos, frente a O(N) clásico. Con 2 qubits basta **1 iteración** para obtener el estado marcado con probabilidad ≈ 1.

**Objetivo:** construir el oráculo, el difusor y verificar la amplificación de amplitud.

**Prerequisito:** Módulos 01 (qubits), 05 (algoritmos).

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
import numpy as np
import matplotlib.pyplot as plt
print('Entorno listo')

## 1. Superposición inicial

Aplicamos Hadamard a ambos qubits para crear la superposición uniforme:
$$|s\rangle = H^{\otimes 2}|00\rangle = \frac{1}{2}(|00\rangle+|01\rangle+|10\rangle+|11\rangle)$$

In [ ]:
qc_init = QuantumCircuit(2)
qc_init.h([0, 1])
sv = Statevector(qc_init)
print('Estado inicial (superposicion uniforme):')
for i, amp in enumerate(sv.data):
    bits = format(i,'02b')
    print(f'  |{bits}>: amplitud={amp.real:.3f}  prob={abs(amp)**2:.3f}')

## 2. Oráculo: marcar el estado |11⟩

El oráculo aplica una fase –1 al estado marcado sin revelar cuál es:
$$\mathcal{O}|x\rangle = (-1)^{f(x)}|x\rangle, \quad f(11)=1, \; f(\text{resto})=0$$

Para marcar |11⟩ basta una puerta **CZ** (controlada-Z).

In [ ]:
# Oraculo para |11>: CZ invierte la fase de |11>
qc_oracle = QuantumCircuit(2, name='Oraculo')
qc_oracle.cz(0, 1)

# Estado tras el oraculo
qc_after_oracle = QuantumCircuit(2)
qc_after_oracle.h([0, 1])
qc_after_oracle.compose(qc_oracle, inplace=True)
sv2 = Statevector(qc_after_oracle)
print('Estado tras el oraculo:')
for i, amp in enumerate(sv2.data):
    bits = format(i,'02b')
    print(f'  |{bits}>: amplitud={amp.real:+.3f}  (|11> tiene fase -1)')

## 3. Difusor: amplificación de amplitud

El difusor $D = 2|s\rangle\langle s| - I$ refleja todos los estados respecto a la media, amplificando el estado marcado.

Circuito del difusor para 2 qubits: H–X–CZ–X–H

In [ ]:
qc_diff = QuantumCircuit(2, name='Difusor')
qc_diff.h([0, 1])
qc_diff.x([0, 1])
qc_diff.cz(0, 1)
qc_diff.x([0, 1])
qc_diff.h([0, 1])

# Circuito completo de Grover (1 iteracion)
qc_grover = QuantumCircuit(2, 2)
qc_grover.h([0, 1])              # Superposicion
qc_grover.compose(qc_oracle, inplace=True)  # Oraculo
qc_grover.compose(qc_diff, inplace=True)    # Difusor
qc_grover.measure([0, 1], [0, 1])

qc_grover.draw('text')

In [ ]:
# Verificar probabilidades con Statevector (sin medicion)
qc_sv = QuantumCircuit(2)
qc_sv.h([0, 1])
qc_sv.compose(qc_oracle, inplace=True)
qc_sv.compose(qc_diff, inplace=True)
sv_final = Statevector(qc_sv)
probs = sv_final.probabilities_dict()
print('Probabilidades finales tras 1 iteracion de Grover:')
for estado, prob in sorted(probs.items(), key=lambda x: -x[1]):
    bar = '█' * int(prob * 40)
    print(f'  |{estado}>: {prob:.4f}  {bar}')
print(f'\nProbabilidad de encontrar |11> (estado marcado): {probs.get("11", 0):.4f}')
print(f'Ventaja cuantica: {probs.get("11", 0) / 0.25:.1f}x sobre busqueda aleatoria')

## 4. Intuición geométrica y escalado

Cada iteración de Grover rota el estado un ángulo 2θ hacia |w⟩ en el plano {|s⟩, |w⟩}. Para N elementos y M marcados, el número óptimo de iteraciones es:
$$k_{\text{opt}} = \left\lfloor \frac{\pi}{4}\sqrt{\frac{N}{M}} \right\rfloor$$

In [ ]:
# Probabilidad de exito vs numero de iteraciones para N=4, 1 marcado
def grover_prob(k, N=4, M=1):
    theta = np.arcsin(np.sqrt(M/N))
    return np.sin((2*k+1)*theta)**2

iteraciones = range(1, 8)
probs_iter = [grover_prob(k) for k in iteraciones]

plt.figure(figsize=(7, 4))
plt.bar(iteraciones, probs_iter, color='steelblue', alpha=0.8, edgecolor='black')
plt.axhline(1.0, color='red', ls='--', lw=1.5, label='Prob=1')
plt.xlabel('Numero de iteraciones k'); plt.ylabel('P(exito)')
plt.title('Grover N=4: probabilidad por numero de iteraciones')
plt.xticks(iteraciones); plt.ylim(0, 1.1); plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('grover_prob.png', dpi=100); plt.show()
print(f'1 iteracion es optima para N=4: P={grover_prob(1):.4f}')